**This notebook is an exercise in the [Intro to Game AI and Reinforcement Learning](https://www.kaggle.com/learn/intro-to-game-ai-and-reinforcement-learning) course.  You can reference the tutorial at [this link](https://www.kaggle.com/alexisbcook/one-step-lookahead).**

---


# Introduction

In the tutorial, you learned how to define a simple heuristic that the agent used to select moves.  In this exercise, you'll check your understanding and make the heuristic more complex.

To get started, run the code cell below to set up our feedback system.

In [ ]:
from learntools.core import binder
binder.bind(globals())
from learntools.game_ai.ex2 import *

### 1) A more complex heuristic

The heuristic from the tutorial looks at all groups of four adjacent grid locations on the same row, column, or diagonal and assigns points for each occurrence of the following patterns:

<center>
<img src="https://storage.googleapis.com/kaggle-media/learn/images/vzQa4ML.png" width=60%><br/>
</center>

In the image above, we assume that the agent is the red player, and the opponent plays yellow discs.

For reference, here is the `get_heuristic()` function from the tutorial:
```python
def get_heuristic(grid, mark, config):
    num_threes = count_windows(grid, 3, mark, config)
    num_fours = count_windows(grid, 4, mark, config)
    num_threes_opp = count_windows(grid, 3, mark%2+1, config)
    score = num_threes - 1e2*num_threes_opp + 1e6*num_fours
    return score
```

In the `get_heuristic()` function, `num_fours`, `num_threes`, and `num_threes_opp` are the number of windows in the game grid that are assigned 1000000, 1, and -100 point(s), respectively. 
    
In this tutorial, you'll change the heuristic to the following (where you decide the number of points to apply in each of `A`, `B`, `C`, `D`, and `E`).  You will define these values in the code cell below.

<center>
<img src="https://storage.googleapis.com/kaggle-media/learn/images/FBoWr2f.png" width=80%><br/>
</center>
    

To check your answer, we use your values to create a heuristic function as follows:

```python
def get_heuristic_q1(grid, col, mark, config):
    num_twos = count_windows(grid, 2, mark, config)
    num_threes = count_windows(grid, 3, mark, config)
    num_fours = count_windows(grid, 4, mark, config)
    num_twos_opp = count_windows(grid, 2, mark%2+1, config)
    num_threes_opp = count_windows(grid, 3, mark%2+1, config)
    score = A*num_fours + B*num_threes + C*num_twos + D*num_twos_opp + E*num_threes_opp
    return score
```

This heuristic is then used to create an agent, that competes against the agent from the tutorial in 50 different game rounds.  In order to be marked correct, 
- your agent must win at least half of the games, and
- `C` and `D` must both be nonzero.

In [ ]:
# Measured, not guessed: these weights beat the tutorial agent in
# 143 of 200 rounds (0.71) locally, sides swapped. See kaggle/connectx/tune_heuristic.py.
#
# A  my four in a row      -- win now, so it has to dominate everything else
# B  my three plus a gap   -- a threat the opponent must answer
# C  my two plus two gaps  -- weak, but it breaks ties towards useful shapes
# D  their two plus gaps   -- small penalty, enough to prefer crowding them
# E  their three plus gap  -- must outweigh B, or the agent builds while it loses
A = 1e+06
B = 100
C = 1
D = -1
E = -10000

# Check your answer (this will take a few seconds to run!)
q_1.check()


In [ ]:
# Lines below will give you a hint or solution code
#q_1.hint()
#q_1.solution()

### 2) Does the agent win?

Consider the game board below.  

<center>
<img src="https://storage.googleapis.com/kaggle-media/learn/images/AlnaQ3J.png" width=30%><br/>
</center>

Say the agent uses red discs, and it's the agent's turn.  
- If the agent uses the heuristic **_from the tutorial_**, does it win or lose the game?
- If the agent uses the heuristic **_that you just implemented_**, does it win or lose the game?

In [ ]:
#q_2.hint()

In [ ]:
# Check your answer (Run this code cell to receive credit!)
q_2.solution()

### 3) Submit to the competition

Now, it's time to submit an agent to the competition!  Use the next code cell to define an agent.  (You can see an example of how to write a valid agent in **[this notebook](https://www.kaggle.com/alexisbcook/create-a-connectx-agent)**.)

You're encouraged to use what you learned in the first question of this exercise to write an agent.  Use the code from the tutorial as a starting point. 

In [ ]:
def my_agent(obs, config):
    import random
    import time

    t_start = time.time()

    try:
        COLS = config.columns
        ROWS = config.rows
        K = config.inarow

        board = obs.board
        me = obs.mark
        valid = [c for c in range(COLS) if board[c] == 0]
        if not valid:
            return 0

        # --- bitboard layout -------------------------------------------
        # bit index = col * H + row, row 0 = bottom. H = ROWS + 1 leaves one
        # always-empty sentinel row per column, which is what stops vertical
        # and diagonal runs from wrapping between columns.
        H = ROWS + 1
        DIRS = (1, H, H + 1, H - 1)

        FULL = 0
        BOTTOM = 0
        COLMASK = []
        for c in range(COLS):
            cm = 0
            for r in range(ROWS):
                cm |= 1 << (c * H + r)
            COLMASK.append(cm)
            FULL |= cm
            BOTTOM |= 1 << (c * H)

        my_pos = 0
        mask = 0
        for c in range(COLS):
            for r in range(ROWS):
                cell = board[(ROWS - 1 - r) * COLS + c]
                if cell:
                    b = 1 << (c * H + r)
                    mask |= b
                    if cell == me:
                        my_pos |= b
        op_pos = mask ^ my_pos

        # Centre columns first: they sit in more winning lines, so they produce
        # cutoffs earlier.
        mid = (COLS - 1) / 2.0
        ORDER = sorted(range(COLS), key=lambda c: abs(c - mid))

        # Shift plans for "which empty squares complete a run of K".
        # For each direction and each position of the gap within the run, the
        # other K-1 cells must already be ours.
        PLANS = []
        for d in DIRS:
            for gap in range(K):
                PLANS.append(tuple((i - gap) * d for i in range(K) if i != gap))

        def shift(x, s):
            return x >> s if s >= 0 else x << -s

        def connected(pos):
            for d in DIRS:
                m = pos
                for i in range(1, K):
                    m &= pos >> (d * i)
                    if not m:
                        break
                if m:
                    return True
            return False

        def threat_squares(pos, msk):
            """Empty cells that would complete a K-run for `pos`."""
            res = 0
            for plan in PLANS:
                m = shift(pos, plan[0])
                for s in plan[1:]:
                    m &= shift(pos, s)
                    if not m:
                        break
                if m:
                    res |= m
            return res & FULL & ~msk

        CENTRE = COLMASK[COLS // 2]
        if COLS % 2 == 0:
            CENTRE |= COLMASK[COLS // 2 - 1]

        def evaluate(my, op, msk):
            my_t = threat_squares(my, msk)
            op_t = threat_squares(op, msk)
            playable = (msk + BOTTOM) & FULL
            score = 0
            score += 16 * bin(my_t).count("1") - 16 * bin(op_t).count("1")
            score += 40 * bin(my_t & playable).count("1")
            score -= 40 * bin(op_t & playable).count("1")
            score += 2 * bin(my & CENTRE).count("1")
            score -= 2 * bin(op & CENTRE).count("1")
            return score

        # --- search -----------------------------------------------------
        WIN = 10 ** 6
        INF = 10 ** 9
        budget = 0.90
        counter = [0]
        tt = {}

        class TimeUp(Exception):
            pass

        def negamax(my, op, msk, depth, alpha, beta, ply):
            counter[0] += 1
            if counter[0] & 511 == 0 and time.time() - t_start > budget:
                raise TimeUp

            nb = msk + BOTTOM
            moves = []
            for c in ORDER:
                b = nb & COLMASK[c]
                if b & FULL:
                    if connected(my | b):
                        return WIN - ply
                    moves.append(b)
            if not moves:
                return 0
            if depth == 0:
                return evaluate(my, op, msk)

            key = (my, msk)
            hit = tt.get(key)
            if hit is not None and hit[0] >= depth:
                _, flag, val = hit
                if flag == 0:
                    return val
                if flag == 1 and val > alpha:
                    alpha = val
                elif flag == 2 and val < beta:
                    beta = val
                if alpha >= beta:
                    return val

            alpha0 = alpha
            best = -INF
            for b in moves:
                v = -negamax(op, my | b, msk | b, depth - 1, -beta, -alpha, ply + 1)
                if v > best:
                    best = v
                if best > alpha:
                    alpha = best
                if alpha >= beta:
                    break

            if best <= alpha0:
                flag = 2
            elif best >= beta:
                flag = 1
            else:
                flag = 0
            tt[key] = (depth, flag, best)
            return best

        # --- root -------------------------------------------------------
        nb0 = mask + BOTTOM
        root_moves = []
        for c in ORDER:
            b = nb0 & COLMASK[c]
            if b & FULL:
                root_moves.append((c, b))

        # Win now, and never hand the opponent a win next move.
        for c, b in root_moves:
            if connected(my_pos | b):
                return c
        for c, b in root_moves:
            if connected(op_pos | b):
                return c

        best_move = root_moves[0][0]
        max_depth = ROWS * COLS - bin(mask).count("1")
        depth = 2
        while depth <= max_depth:
            try:
                alpha = -INF
                local_best = None
                for c, b in root_moves:
                    v = -negamax(op_pos, my_pos | b, mask | b, depth - 1, -INF, -alpha, 1)
                    if local_best is None or v > alpha:
                        alpha = v
                        local_best = c
                if local_best is not None:
                    best_move = local_best
                if alpha >= WIN - depth:
                    break
            except TimeUp:
                break
            if time.time() - t_start > budget * 0.5:
                break
            depth += 1

        return int(best_move)

    except Exception:
        try:
            return int(random.choice([c for c in range(config.columns) if obs.board[c] == 0]))
        except Exception:
            return 0


In [ ]:
# Run this code cell to get credit for creating an agent
q_3.check()

Run the next code cell to convert your agent to a submission file.

In [ ]:
import inspect
import os

def write_agent_to_file(function, file):
    with open(file, "a" if os.path.exists(file) else "w") as f:
        f.write(inspect.getsource(function))
        print(function, "written to", file)

write_agent_to_file(my_agent, "submission.py")

Then, follow these steps to submit your agent to the competition:
1. Begin by clicking on the **Save Version** button in the top right corner of the window.  This will generate a pop-up window.  
2. Ensure that the **Save and Run All** option is selected, and then click on the **Save** button.
3. This generates a window in the bottom left corner of the notebook.  After it has finished running, click on the number to the right of the **Save Version** button.  This pulls up a list of versions on the right of the screen.  Click on the ellipsis **(...)** to the right of the most recent version, and select **Open in Viewer**.  This brings you into view mode of the same page. You will need to scroll down to get back to these instructions.
4. Click on the **Data** tab near the top of the screen.  Then, click on the file you would like to submit, and click on the **Submit** button to submit your results to the leaderboard.

You have now successfully submitted to the competition!

If you want to keep working to improve your performance, select the **Edit** button in the top right of the screen. Then you can change your code and repeat the process. There's a lot of room to improve, and you will climb up the leaderboard as you work.


Go to **"My Submissions"** to view your score and episodes being played.

# Keep going

Move on to **[develop a longer-term strategy](https://www.kaggle.com/alexisbcook/n-step-lookahead)** with the minimax algorithm.

---




*Have questions or comments? Visit the [course discussion forum](https://www.kaggle.com/learn/intro-to-game-ai-and-reinforcement-learning/discussion) to chat with other learners.*